# 02 - Backtest a trained run

Pick a strategy, edit its knobs and the costs, press **Run backtest**. Everything is replayed on the
run's own TEST block (out of sample: the purged split is rebuilt from the run's config) with
next-open fills, stops on the bar's high/low, fees + spread + slippage, and three baselines
(buy-and-hold, always-flat, random entries at the same frequency).

In [1]:
# Parameters
RUN_DIR = None                  # a runs/<id> directory; None -> the newest run under RUNS_DIR
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"

In [2]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from neural_trade.notebook import BacktestExplorer, pick_run
from neural_trade.strategy import Strategies

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
explorer = BacktestExplorer.from_run(run_dir, csv_path=CSV_PATH)
display(explorer.widget())
explorer.click_run()   # render the default strategy once; then use the controls

run: ..\runs\20260924T114819Z-e23fd9f-dirty-af67ee43


CalibrationPipeline loaded from '..\runs\20260924T114819Z-e23fd9f-dirty-af67ee43\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


Static copy of that first result (the widget above holds the live one):

In [3]:
from neural_trade.registries.visualizations import Visualizations

display(explorer.summary_frame().round(4))
Visualizations.build("plotly_trading", explorer.last, explorer.config, bars=explorer.bars).show()

,n_trades,total_return,sharpe_net,max_drawdown,hit_rate,hit_rate_gross,profit_factor,exposure,fees_paid,costs_paid,random percentile (return)
calibrated_quantile,144.0,-0.2766,-80.3341,0.2772,0.1319,0.5347,0.0864,0.1833,2483.8496,3229.0045,95.0
buy_and_hold,1.0,0.0410,6.6470,0.0495,1.0000,1.0000,inf,0.9999,20.4362,26.5671,NaN
always_flat,0.0,0.0000,0.0000,0.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,NaN
random same freq (mean of 20),NaN,-0.3108,-102.0483,NaN,NaN,NaN,NaN,NaN,NaN,NaN,95.0


## Every strategy with its default knobs

In [4]:
pd.DataFrame({name: explorer.run(name, costs={"random_seeds": 0}, baselines=False).summary
              for name in Strategies.list_names()}).T[["n_trades", "total_return", "sharpe_net", "max_drawdown",
                                                      "hit_rate", "hit_rate_gross", "profit_factor", "exposure", "gross_pnl",
                                                      "costs_paid"]]

,n_trades,total_return,sharpe_net,max_drawdown,hit_rate,hit_rate_gross,profit_factor,exposure,gross_pnl,costs_paid
always_flat,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000
buy_and_hold,1.0,0.040967,6.646960,0.049498,1.000000,1.000000,inf,0.999862,436.235136,26.567106
calibrated_quantile,144.0,-0.276648,-80.334129,0.277209,0.131944,0.534722,0.086407,0.183250,462.520555,3229.004503
enhanced_multi_horizon,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000
liberal,67.0,-0.165983,-74.948945,0.165983,0.000000,0.358209,0.000000,0.030127,-67.144689,1592.684823
random_signal,243.0,-0.470051,-130.015563,0.470846,0.069959,0.555556,0.031570,0.335821,-80.726761,4619.779058
threshold_spike,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,0.000000,0.000000,0.000000


A strategy with 0 trades is usually blocked by one of two things. Fixed probability lines
(`threshold_spike` enters above 0.65 / below 0.35) are rarely crossed by calibrated probabilities.
Strategies that need the predicted move to agree with the side (`enhanced_multi_horizon`) cannot trade
when delta shrinkage serves a zero delta. Delta shrinkage sets beta = 0 when the raw price head's
moves pointed the wrong way on the calibration block (negative correlation with the realised move). This run's values:

In [5]:
import numpy as np

p_up = explorer.signals.p
pd.DataFrame({"delta beta (served delta = beta x raw)": pd.Series(explorer.blocks["predictor"].bundle.calibration_pipeline.delta_scale),
              "P(up) 1st percentile": np.percentile(p_up, 1, axis=0),
              "P(up) 99th percentile": np.percentile(p_up, 99, axis=0)}, index=["h0", "h1", "h2"]).round(4)

,delta beta (served delta = beta x raw),P(up) 1st percentile,P(up) 99th percentile
h0,0.0000,0.4164,0.5819
h1,0.0000,0.3997,0.5751
h2,0.0608,0.4088,0.5844
